# Cross-docking sampling (dummy data)

This notebook runs **cross-docking** on **notebooks/dummy_data**: for each reference complex (folder `PDB_LIG`), we dock **query ligands** from other complexes (`query_<SOURCE>.sdf`) into that pocket. The pocket and CoM are defined using the **reference** ligand (`*_ligand.sdf`).

**Setup:** Ensure `dummy_data` has been populated with query SDFs (run `dummy_data/setup_crossdock_queries.py`). The experiment config **dummy_crossdock** uses `sdf_regex: query_.*\.sdf` and `ref_sdf_regex: .*_ligand\.sdf`. Run from **project root**.

In [19]:
import os
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
from pytorch_lightning import seed_everything
from torch_geometric.data import Batch

from sigmadock.config import get_experiment_config
from sigmadock.data import SigmaDataset
from sigmadock.datafronts import MetaFront
from sigmadock.diff.sampling import sample_notebook
from sigmadock.utils import load_from_scratch

# Project root (run notebook from repo root so notebooks/dummy_data exists)
ROOT = Path.cwd().resolve()
DUMMY_DATA = ROOT / "dummy_data"
if not DUMMY_DATA.exists():
    raise SystemExit(f"Expected {DUMMY_DATA}. Run from project root.")

# NOTE: Hardcoding checkpoint path for testing.
os.environ["SIGMADOCK_CKPT_DIR"] = "/vols/ziz/not-backed-up/scratch/sigmadock/paper/model_0.ckpt"
CKPT_PATH = Path(os.environ.get("SIGMADOCK_CKPT_DIR", "checkpoints/last.ckpt")).resolve()
CACHE = ROOT / ".cache_igso3"

## DataFront and dataset

**dummy_crossdock** (conf/experiments/dummy_crossdock.yaml): `query_*.sdf` = ligands to dock, `*_ligand.sdf` = reference for pocket/CoM, `*.pdb` = protein.

In [20]:
config = get_experiment_config("dummy_crossdock", ROOT)
print(f"Dataset: {config.dataset}")
print(f"sdf_regex (queries): {config.sdf_regex}")
print(f"ref_sdf_regex (reference): {config.ref_sdf_regex}")

datafront = MetaFront([config])
print(f"DataFront: {len(datafront)} (query, protein, ref) pairs")

dataset = SigmaDataset(
    datafront=datafront,
    use_esm_embeddings=False,
    sample_conformer=False,
    alignment_tries=0,
    skip_bounds_check=True,
    prot_coordinate_distance_noise=0,
    pocket_com_noise = 1/4,
    pocket_distance_noise = 1/4,
    get_mol_info=True,
    force_retry=False,
    seed=123,
    verbose=True,
)
print(f"Dataset length: {len(dataset)}")

INFO:posebusters.posebusters:Using default configuration for mode redock_fast.


Dataset: /vols/ziz/not-backed-up/prat/local/AlphaDock/notebooks/dummy_data
sdf_regex (queries): query_.*\.sdf$
ref_sdf_regex (reference): .*_ligand\.sdf$
DataFront: 90 (query, protein, ref) pairs
Dataset length: 90


## Load denoiser and run sampling

Load checkpoint (or dummy model), then run multi-seed sampling on a batch of cross-docking pairs. Each pair is (query ligand SDF, protein PDB, reference ligand SDF).

In [3]:
if CKPT_PATH.exists():
    model = load_from_scratch(
        CKPT_PATH,
        enforced_cfg={"cache_path": str(CACHE)},
        load_ema=True,
        strict=True,
    )
    denoiser = model.ema_model.model
else:
    print("No checkpoint found, using dummy model")
    import torch.nn as nn
    from sigmadock.diff.denoiser import SigmaDockDenoiser
    _dummy = nn.Module()
    _dummy.model = nn.Embedding(1, 10)
    denoiser = SigmaDockDenoiser(model=_dummy, include_interactions=False, cache_path=CACHE)

denoiser.eval()
denoiser.to("cuda" if torch.cuda.is_available() else "cpu")
device = next(denoiser.parameters()).device
denoiser.diffuser._so3_diffuser.set_device(device)
print(f"Denoiser on {device}")

Error loading checkpoint with torch.load: No module named 'alphadock'. Attempting load with legacy module alias.
[WARNING] Using provided HPARAMS that could be modified from oracle.py
[WARN] Ignored kwargs in EquiformerV2: {'cache_path': '/vols/ziz/not-backed-up/prat/local/AlphaDock/notebooks/.cache_igso3'}. Please check for typos unless unintended.
[INFO] DIST | Skipping inter_fragments edges as ligand_ligand_interactions is False
Using cached IGSO3 in /vols/ziz/not-backed-up/prat/local/AlphaDock/notebooks/.cache_igso3/eps_1000_omega_2000_min_sigma_0_1_max_sigma_1_5_schedule_logarithmic
Successfully loaded EMA model.
Denoiser on cuda:0


In [21]:
# Using core Batch Size = 9 Which is exactly the number of query complexes per PDB entry in DUMMY DATA
BATCH_SIZE = 9 * 4
NUM_SEEDS = 1
SEED_BASE = 123

seed_everything(SEED_BASE, workers=True, verbose=False)
indices = list(range(min(BATCH_SIZE, len(dataset))))
batch_list = [dataset[i] for i in indices]
batch = Batch.from_data_list(batch_list)
batch = batch.to("cuda")

all_outputs = []
seeds = np.random.randint(0, 1000, size=NUM_SEEDS).tolist()
for i, seed in enumerate(seeds):
    out = sample_notebook(
        denoiser=denoiser,
        batch=deepcopy(batch),
        t_min=2e-3,
        num_steps=30,
        discretization="edm",
        rho=3,
        solver="euler",
        noise_scale=0,
        noise_decay=1,
        seed=seed,
        use_true_scores=False,
        verbose=(i == 0),
    )
    all_outputs.append(out)

print(f"Sampled {NUM_SEEDS} seeds for {len(batch_list)} cross-docking pairs.")

Using 30 steps for reverse sampling with: 
             seed=468 
             rho=3 
             solver=euler 
             noise_scale=0 
             t_min=0.002 
             


29it [00:28,  1.01it/s]

Sampled 1 seeds for 36 cross-docking pairs.


In [42]:
import ipywidgets as widgets
import plotly.graph_objs as go
from IPython.display import display

from sigmadock.geo.viz import plot_interaction_graph_3d_plotly
from sigmadock.torch_utils.utils import re_batch_with_attrs

# Fix Camera Angle
camera_angle = {
    'center': {'x': 0, 'y': 0, 'z': 0},
    'eye': {'x': -0.7465395149035515, 'y': -0.9797237326004345, 'z': -0.4169740241924182},
    'projection': {'type': 'perspective'},
    'up': {'x': 0.8020677951794761, 'y': -0.596196149887429, 'z': -0.03517673656467074}
}

# Set BATCH_IDX according to first entry. Visualizing 3 for the same Pocket with same Camera Pose.
STEP = -1
PDB_IDX = 2
BATCH_IDX = 0 + 9 * PDB_IDX
SEED_IDX = 0

sample_batch, sample_pos, sample_edges = all_outputs[SEED_IDX][0], all_outputs[SEED_IDX][1], all_outputs[SEED_IDX][2]
sample_batch.pos_t = torch.Tensor(sample_pos[STEP])
batch_out = re_batch_with_attrs(sample_batch, ["pos_t", "pos_0"])

fw = go.FigureWidget()
plot_interaction_graph_3d_plotly(
    batch_out[BATCH_IDX].cpu(),
    pos_key="pos_t",
    show_protein=True,
    show_protein_virtual=True,
    show_ligand_virtual=False,
    show_overconstrained=True,
    show_triangulation=False,
    show_interaction_edges=False,
    fig=fw,
    camera_angle=camera_angle,
)

if fw is not None:
    button = widgets.Button(description="Get Camera Angle")
    output = widgets.Output()

    def on_button_click(b): # noqa
        with output:
            output.clear_output()
            print("Copy this dictionary into your code:")
            # This reads the camera settings directly from the plot of your molecule
            print(fw.layout.scene.camera)

    button.on_click(on_button_click)
    print("Rotate the molecule below, then click the button to get the camera settings.")
    display(widgets.VBox([fw, button, output]))

Rotate the molecule below, then click the button to get the camera settings.


    'data': [{'hoverinfo': 'none',
              'line': {'color': '#2E91E5', 'w…

In [43]:
fw = go.FigureWidget()
plot_interaction_graph_3d_plotly(
    batch_out[BATCH_IDX + 1].cpu(),
    pos_key="pos_t",
    show_protein=True,
    show_protein_virtual=True,
    show_ligand_virtual=False,
    show_overconstrained=True,
    show_triangulation=False,
    show_interaction_edges=False,
    fig=fw,
    camera_angle=camera_angle,
)

FigureWidget({
    'data': [{'hoverinfo': 'none',
              'line': {'color': '#2E91E5', 'width': 6},
              'mode': 'lines',
              'name': 'Sample frag_0',
              'opacity': 0.95,
              'showlegend': False,
              'type': 'scatter3d',
              'uid': 'b3ea3c42-7ff2-40e0-a177-480730e0715e',
              'x': [1.3255032300949097, 1.0438400506973267, None],
              'y': [-0.21400612592697144, 0.1118650883436203, None],
              'z': [1.320746660232544, 1.0276169776916504, None]},
             {'hoverinfo': 'none',
              'line': {'color': '#2E91E5', 'width': 6},
              'mode': 'lines',
              'name': 'Sample frag_0',
              'opacity': 0.95,
              'showlegend': False,
              'type': 'scatter3d',
              'uid': 'cf234105-3855-454c-9f18-c38e2eb4cfda',
              'x': [1.0438400506973267, 1.3255032300949097, None],
              'y': [0.1118650883436203, -0.21400612592697144, None],


In [44]:
fw = go.FigureWidget()
plot_interaction_graph_3d_plotly(
    batch_out[BATCH_IDX + 2].cpu(),
    pos_key="pos_t",
    show_protein=True,
    show_protein_virtual=True,
    show_ligand_virtual=False,
    show_overconstrained=True,
    show_triangulation=False,
    show_interaction_edges=False,
    fig=fw,
    camera_angle=camera_angle,
)

FigureWidget({
    'data': [{'hoverinfo': 'none',
              'line': {'color': '#2E91E5', 'width': 6},
              'mode': 'lines',
              'name': 'Sample frag_0',
              'opacity': 0.95,
              'showlegend': False,
              'type': 'scatter3d',
              'uid': '44a375a6-aca4-4d3d-b1b1-cfe22e5c0fc1',
              'x': [1.5234122276306152, 1.3022747039794922, None],
              'y': [0.45754536986351013, -0.03474727272987366, None],
              'z': [1.768662691116333, 1.5921356678009033, None]},
             {'hoverinfo': 'none',
              'line': {'color': '#2E91E5', 'width': 6},
              'mode': 'lines',
              'name': 'Sample frag_0',
              'opacity': 0.95,
              'showlegend': False,
              'type': 'scatter3d',
              'uid': '668f7106-2b0a-4f28-855c-57249856b75e',
              'x': [1.3022747039794922, 1.5234122276306152, None],
              'y': [-0.03474727272987366, 0.45754536986351013, None]

### Extra Fragment Viz 

In [ ]:
from sigmadock.chem.pyviz import view_pocket_fragments
from sigmadock.chem.fragmentation import get_fragments_as_mols
import copy

data = dataset[4]
view_pocket_fragments(
    # NOTE: removing reference ligand for now by refering the pocket as the starting ligand.
    copy.deepcopy(data.mol_info["original"]),
    fragments = get_fragments_as_mols(copy.deepcopy(data.mol_info["fragmented"])),
    fragment_opacity=1.0,
    pocket_style="stick",
    fragment_style="stick",
    pocket_surface_opacity=0.4,
    ref_ligand_opacity=0.6,
    pocket_alpha=0.4,
    reference_color="green",
    pocket_surface_color="gray",
    pocket_surface=False,
    width=600,
    height=500,
    view_dummies=False,
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.